<a href="https://colab.research.google.com/github/flathfk/Bootcamp_Study/blob/main/260309_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Visualization을 위한 JSON 기반 테이블 생성 연습

## [실습 과제] JavaScript를 이용한 공항 운항 정보 시스템(FIDS) 구현

### 1. 과제 개요

본 과제는 외부 JavaScript 파일(`flight.js`)에 저장된 항공 운항 데이터를 읽어와서, 인천국제공항 출발 현황판 스타일의 웹 페이지를 동적으로 구성하는 것을 목표로 합니다.

---

### 2. 준비 파일

실습을 위해 다음 두 개의 파일이 동일한 폴더에 있어야 합니다.

- **flightInfo.html**: 전체 레이아웃 및 렌더링 로직 (강의노트 3-1 [실습 1] CSS 기반의 여신 대출 현황 대시보드 UI 구현 참고)
- **flight.js**: 항공 데이터 배열 (아래 샘플 데이터 참고)
- 기능은 같게 스타일은 최대한 유사하게 만들어 보세요

### 3. 주요 실습 포인트 및 요구사항

### **Step 1: CSS 변수와 다크 모드 레이아웃**

- `:root`를 사용하여 공항 테마 색상(골드, 다크 그레이, 상태별 강조색)을 정의하세요.
- `table`의 `border-collapse` 속성을 활용하여 깔끔한 표 구조를 만드세요.

### **Step 2: 로고 이미지 비율 유지 (이미지 처리)**

- 항공사 로고가 원형이 아닌 **원본 비율**로 출력되도록 설정하세요.
- **핵심 속성**: `object-fit: contain;`을 사용하여 이미지가 찌그러지지 않게 하세요.

### **Step 3: 데이터 바인딩 및 동적 렌더링**

- `flightData.map()` 함수를 사용하여 테이블 행(`tr`)을 생성하세요.
- `schedule`(예정)과 `estimated`(변경) 시간이 다를 경우, 예정 시간에 **취소선**을 긋는 조건부 스타일을 적용하세요.

In [28]:
!pkill -f "node /content/server.js"
!nohup node /content/server.js > /content/server.log 2>&1 &
!tail -n 5 /content/server.log



---



In [29]:
!mkdir -p /content/templates /content/static
!npm init -y
!npm install express

Wrote to /content/package.json:

{
  "name": "content",
  "version": "1.0.0",
  "main": "index.js",
  "scripts": {
    "test": "echo \"Error: no test specified\" && exit 1",
    "start": "node server.js"
  },
  "keywords": [],
  "author": "",
  "license": "ISC",
  "dependencies": {
    "express": "^5.2.1"
  },
  "devDependencies": {},
  "description": ""
}



⠙⠙⠹⠸⠼⠴
up to date, audited 66 packages in 800ms
⠴
⠴22 packages are looking for funding
⠴  run `npm fund` for details
⠴
found 0 vulnerabilities
⠴

In [30]:
%%writefile /content/server.js
const express = require("express");
const path = require("path");
const app = express();

app.use(express.static("/content/static"));  // 절대경로로 변경

app.get("/", (req, res) => {
    res.sendFile("/content/templates/flightInfo.html");  // 절대경로로 변경
});

app.listen(3000, () => {
    console.log("Server running on port 3000");
});

Overwriting /content/server.js


In [31]:
%%writefile /content/templates/flightInfo.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>DEPARTURES</title>
    <style>
        /* 전체에서 재사용할 색상 변수 모음 */
        :root {
            --bg-main: #0d0d0d;         /* 페이지 배경 */
            --bg-table: #252525;        /* 테이블 행 배경 */
            --bg-header: #1a1a2e;       /* 테이블 헤더 행 배경 */
            --gold: #f0b429;            /* 골드 강조색 */
            --text-dim: #888;           /* 흐린 텍스트 (취소선 시간 등) */
            --status-delay: #ff6b6b;    /* 지연 - 빨강 */
            --status-boarding: #51cf66; /* 탑승중 - 초록 */
            --status-done: #888;        /* 출발완료 - 회색 */
        }
        body {
            background-color: var(--bg-main);
            color: white;
            font-family: 'Arial', sans-serif;
            padding: 20px;
        }
        .header-area {
            border-bottom: 2px solid var(--gold);
            padding-bottom: 16px;
            margin-bottom: 24px;
        }
        h1 { color: var(--gold); }
        #clock { color: var(--text-dim); margin: 8px 0; }
        table {
            width: 100%;
            border-collapse: collapse;
        }
        th {
            color: var(--gold);
            text-align: left;
            padding: 14px 16px;
            border-bottom: 1px solid var(--gold);
            background-color: var(--bg-header);
            vertical-align: middle;
        }
        td {
            padding: 16px 16px;
            border-bottom: 1px solid #333;
            background-color: var(--bg-table);
            vertical-align: middle;
        }
        /* 로고 이미지 — object-fit: contain으로 원본 비율 유지 */
        .airline-logo {
            height: 30px;
            width: 80px;
            object-fit: contain;
            vertical-align: middle;
            margin-right: 8px;
        }
        /* 편명 골드 강조 */
        .flight-id {
            color: var(--gold);
            font-weight: bold;
        }
        /* 상태 뱃지 공통 스타일 */
        .badge {
            display: inline-block;
            padding: 4px 12px;
            border-radius: 4px;
            font-size: 13px;
            font-weight: bold;
            border: 1px solid;
        }
        /* 상태별 색상만 개별 지정 */
        .badge-delay    { color: var(--status-delay);    border-color: var(--status-delay); }
        .badge-boarding { color: var(--status-boarding); border-color: var(--status-boarding); }
        .badge-done     { color: var(--status-done);     border-color: var(--status-done); }
    </style>
</head>
<body>
    <div class="container">
        <div class="header-area">
            <h1>✈ 인천국제공항 출발 현황</h1>
            <div id="clock"></div>
        </div>
        <table id="flight-table">
            <thead>
                <tr>
                    <th>시간(예정)</th>
                    <th>항공사</th>
                    <th>편명</th>
                    <th>목적지</th>
                    <th>탑승구</th>
                    <th>상태</th>
                </tr>
            </thead>
            <tbody id="flight-tbody"></tbody>
        </table>
    </div>
    <script>
        const flightData = [
            { "flightId": "KE101", "airline": "대한항공",    "destination": "도쿄/나리타",   "schedule": "09:15", "estimated": "09:30", "gate": "231", "status": "지연",    "logo": "KOREAN.jpg" },
            { "flightId": "OZ202", "airline": "아시아나항공", "destination": "로스앤젤레스",  "schedule": "10:40", "estimated": "10:40", "gate": "112", "status": "출발완료", "logo": "ASIANA.jpg" },
            { "flightId": "LJ055", "airline": "진에어",       "destination": "방콕/수완나품", "schedule": "11:20", "estimated": "11:20", "gate": "105", "status": "탑승중",  "logo": "JIN.jpg" },
            { "flightId": "TW661", "airline": "티웨이항공",   "destination": "다낭",          "schedule": "13:30", "estimated": "14:10", "gate": "124", "status": "지연",    "logo": "TWAY.jpg" }
        ];

        /* 시계 — 1초마다 현재 시각 업데이트 */
        function updateClock() {
            const now = new Date();
            document.getElementById('clock').innerText = now.toLocaleString('ko-KR');
        }
        updateClock();
        setInterval(updateClock, 1000);

        /* map()으로 flightData 배열을 tr 문자열로 변환 후 tbody에 삽입 */
        const tbody = document.getElementById('flight-tbody');
        tbody.innerHTML = flightData.map(flight => {

            // 상태값에 따라 뱃지 클래스 결정
            const badgeClass =
                flight.status === '지연'   ? 'badge-delay' :
                flight.status === '탑승중' ? 'badge-boarding' :
                'badge-done';

            return `<tr>
                <td>
                    <!-- schedule != estimated면 취소선 + 빨간색 변경시간 표시 -->
                    <div style="color: var(--text-dim); text-decoration: ${flight.schedule !== flight.estimated ? 'line-through' : 'none'}; font-size: 13px;">${flight.schedule}</div>
                    <div style="color: ${flight.schedule !== flight.estimated ? 'var(--status-delay)' : 'white'}">${flight.estimated}</div>
                </td>
                <td>
                    <!-- express.static은 /파일명으로 서빙 — /static/ 붙이면 안 됨 -->
                    <img src="/${flight.logo}" class="airline-logo">
                    ${flight.airline}
                </td>
                <td><span class="flight-id">${flight.flightId}</span></td>
                <td>${flight.destination}</td>
                <td>${flight.gate}</td>
                <td><span class="badge ${badgeClass}">${flight.status}</span></td>
            </tr>`;
        }).join('');
    </script>
</body>
</html>

Overwriting /content/templates/flightInfo.html


In [32]:
!nohup node /content/server.js > /content/server.log 2>&1 &
!tail -n 5 /content/server.log

In [33]:
!npm install -g cloudflared
!nohup cloudflared tunnel --url http://localhost:3000 > /content/tunnel.log 2>&1 &

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼
changed 1 package in 1s
⠼

In [37]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" /content/tunnel.log | tail -n 1

https://topic-circles-nor-teachers.trycloudflare.com




---



##PNG 이미지를 JPG로 바꾸기 (배경 불투명화)

In [36]:
from PIL import Image

for filename in ['KOREAN.jpg', 'ASIANA.jpg', 'JIN.jpg', 'TWAY.jpg']:
    img = Image.open(f'/content/static/{filename}').convert('RGBA')
    white_bg = Image.new('RGBA', img.size, (255, 255, 255, 255))
    white_bg.paste(img, mask=img.split()[3])
    white_bg.convert('RGB').save(f'/content/static/{filename}')

print("완료")

완료
